# P4-A1 — Persistent Memory (agents that remember)

Your agents so far have been **amnesiac**: every `invoke` starts from nothing. A real assistant remembers the conversation. LangGraph adds memory with a **checkpointer** — it saves the graph's state after each step, keyed by a **`thread_id`**. Same thread → same conversation; new thread → fresh start.

You'll prove three things: (1) no memory = forgets, (2) a checkpointer + thread_id = remembers, (3) a different thread_id = isolated conversation (this is how one deployment serves many users).

In [ ]:
# --- Setup (given) ---
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langgraph.graph import StateGraph, START, MessagesState
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.checkpoint.memory import InMemorySaver

load_dotenv()
llm = ChatOpenAI(model='gpt-4o-mini')

@tool
def add(a: float, b: float) -> float:
    """Add two numbers."""
    return a + b

TOOLS = [add]

def build_agent(checkpointer=None):
    """Your P2-A7 graph. If a checkpointer is passed, the agent gains memory."""
    bound = llm.bind_tools(TOOLS)
    def agent_node(state):
        return {'messages': [bound.invoke(state['messages'])]}
    g = StateGraph(MessagesState)
    g.add_node('agent', agent_node)
    g.add_node('tools', ToolNode(TOOLS))
    g.add_edge(START, 'agent')
    g.add_conditional_edges('agent', tools_condition)
    g.add_edge('tools', 'agent')
    return g.compile(checkpointer=checkpointer)   # <-- memory plugs in HERE

def say(agent, text, config=None):
    """Send one user turn, return the agent's text reply."""
    out = agent.invoke({'messages': [{'role': 'user', 'content': text}]}, config=config)
    return out['messages'][-1].content

print('ready')

## Task 1 — No memory = it forgets
Build an agent with **no** checkpointer. Send two turns: first tell it something (*"My name is Shubham and my favourite language is Python."*), then ask *"What's my name and favourite language?"*
<br>Show it canNOT answer the second — each `invoke` is independent, so it never saw turn 1.
<br>Hint: `agent = build_agent()`; two separate `say(agent, ...)` calls, no config.

In [ ]:
# TODO: no-memory agent; two turns; show it forgets turn 1


## Task 2 — A checkpointer + thread_id = it remembers
Build an agent **with** an `InMemorySaver`, and invoke it with a `config` carrying a `thread_id`. Run the same two turns on the **same thread_id** — now the follow-up should correctly recall your name and language.
<br>Hint: `agent = build_agent(InMemorySaver())`; `config = {'configurable': {'thread_id': 'user-1'}}`; pass that same `config` to both `say(...)` calls.

In [ ]:
# TODO: memory agent; same thread_id across two turns; show it remembers


## Task 3 — A different thread_id = a separate conversation
Using the **same agent object** from Task 2, ask *"What's my name?"* but with a **different** `thread_id` (e.g. `'user-2'`). Show it does NOT know — that thread has its own fresh memory.
<br>Goal: this is how one deployed agent serves many users without mixing their conversations.

In [ ]:
# TODO: same agent, NEW thread_id; show the memory is isolated per thread


## Task 4 — Explain (in your own words)
1. What does the checkpointer actually store, and what role does `thread_id` play?
2. `InMemorySaver` loses everything when the process restarts. For a real deployed app, what would you swap it for, and why does that matter? (Think about your P2-A7 note on persistence.)

> _your answer here_